In [2]:
import pandas as pd
from scipy.io import mmread

genes = pd.read_csv("data/EoE_gene_processed.tsv", sep="\t", header=None, encoding='latin-1')
cells = pd.read_csv("data/EoE_cell_processed.tsv", sep="\t", header=None, encoding='latin-1')
meta = pd.read_csv("data/EoE_meta.txt", sep="\t", encoding='latin-1')

print(genes.shape)
print(cells.shape)
print(meta.shape)

print(genes.head())
print(cells.head())
print(meta.head())
print(meta.columns)

(33694, 1)
(393763, 1)
(393764, 14)
              0
0  RP11-34P13.3
1       FAM138A
2         OR4F5
3  RP11-34P13.7
4  RP11-34P13.8
                        0
0  E1881_AAACCTGAGAATCTCC
1  E1881_AAACCTGAGACTGTAA
2  E1881_AAACCTGAGGAACTGC
3  E1881_AAACCTGAGGTGATTA
4  E1881_AAACCTGAGTCGAGTG
                     NAME biosample_id        disease  \
0                    TYPE        group          group   
1  E1881_AAACCTGAGAATCTCC        E1881  MONDO:0005361   
2  E1881_AAACCTGAGACTGTAA        E1881  MONDO:0005361   
3  E1881_AAACCTGAGGAACTGC        E1881  MONDO:0005361   
4  E1881_AAACCTGAGGTGATTA        E1881  MONDO:0005361   

    disease__ontology_label disease_status donor_id  \
0                     group          group    group   
1  eosinophilic esophagitis      Remission    E1881   
2  eosinophilic esophagitis      Remission    E1881   
3  eosinophilic esophagitis      Remission    E1881   
4  eosinophilic esophagitis      Remission    E1881   

  library_preparation_protocol library

In [3]:
meta = meta.iloc[1:].copy()   # ignore the first row type/group. 
meta["NAME"] = meta["NAME"].astype(str)

genes_list = genes[0].astype(str).tolist()
cells_list = cells[0].astype(str).tolist()

print(len(genes_list), len(cells_list), meta.shape)
print(meta["NAME"].head())
print(sum(meta["NAME"].isin(cells_list)))
print(len(cells_list))

33694 393763 (393763, 14)
1    E1881_AAACCTGAGAATCTCC
2    E1881_AAACCTGAGACTGTAA
3    E1881_AAACCTGAGGAACTGC
4    E1881_AAACCTGAGGTGATTA
5    E1881_AAACCTGAGTCGAGTG
Name: NAME, dtype: object
393763
393763


In [5]:
import numpy as np # strafitying the data for balanced sampling
np.random.seed(42)
sampled = []
for (ct, ds), group in meta.groupby(['cell_type_anno', 'disease_status']):
    target_n = max(50, int(len(group) * 0.20))
    n = min(len(group), target_n)
    
    # n = min(len(group), 50)  # max 50
    sampled.append(group.sample(n, random_state=42))

meta_sub = pd.concat(sampled)
print(f"Total: {len(meta_sub)}")
print(meta_sub['disease_status'].value_counts())
print(meta_sub['cell_type_anno'].value_counts())

Total: 80909
disease_status
Active       43599
Remission    19234
Ctrl         18076
Name: count, dtype: int64
cell_type_anno
Suprabasal                  16647
Apical cell                 13712
Basal cell (cycling)        12319
CD8+ Teff (KLRC1)            8830
Quiescent basal cell         6840
Mast cell                    5486
Treg                         2029
Capillary (RGCC)             1663
Fibroblast                   1352
Venous (ACKR1)               1178
Th17                         1132
Pericyte                      962
Eosinophil                    726
Erythroid cell                602
Th2                           580
CD8+ Trm (GZMK)               434
Arterial (GJA4)               374
CD4+ Trm (CCL5)               373
Plasma cell (IgA+)            361
Mast cell (cycling)           342
Macrophage (SEPP1)            301
NK (XCL1/CD160)               299
CD4+ Tcm                      268
cDC2B (CD1C)                  243
Macrophage (PLAC8)            227
CD8+ T cell (MHC II)    

In [6]:
from scipy.io import mmread

mat = mmread("data/EoE-processed.mtx").tocsc()
print(mat.shape)

(33694, 393763)


In [7]:
target_cells = meta_sub['NAME'].tolist()
cell_idx = [i for i, c in enumerate(cells_list) if c in set(target_cells)]
print(f"Found {len(cell_idx)} cells")

# 切片
mat_sub = mat[:, cell_idx]  # genes × cells
cells_sub = [cells_list[i] for i in cell_idx]

# 对齐 meta 顺序
meta_aligned = meta_sub.set_index('NAME').loc[cells_sub]

KeyboardInterrupt: 

In [ ]:
import anndata as ad
adata = ad.AnnData(
    X=mat_sub.T,  # cells × genes
    obs=meta_aligned,
    var=pd.DataFrame(index=genes_list)
)

print(adata)
adata.write_h5ad("eoe_subset_8w.h5ad")
matrix_bytes = mat.data.nbytes + mat.indices.nbytes + mat.indptr.nbytes
print(matrix_bytes / 1024**3, "GB")

AnnData object with n_obs × n_vars = 7212 × 33694
    obs: 'biosample_id', 'disease', 'disease__ontology_label', 'disease_status', 'donor_id', 'library_preparation_protocol', 'library_preparation_protocol__ontology_label', 'organ', 'organ__ontology_label', 'sex', 'species', 'species__ontology_label', 'cell_type_anno'
disease_status
Active       2701
Remission    2286
Ctrl         2225
Name: count, dtype: int64
cell_type_anno
Gamma delta T cell          150
Fibroblast                  150
CD8+ Teff (KLRC1)           150
Pericyte                    150
Basal cell (cycling)        150
Macrophage (cycling)        150
Mast cell                   150
CD4+ Tcm                    150
CD8+ T cell (MHC II)        150
Macrophage (SEPP1/IL1B)     150
Capillary (RGCC)            150
Mature DC (FSCN1)           150
NK (XCL1/CD160)             150
Treg                        150
Macrophage (PLAC8)          150
Th2                         150
cDC2B (CD1C)                150
B cell (memory)            

In [7]:
adata.write_h5ad("eoe_subset_7k.h5ad")
matrix_bytes = mat.data.nbytes + mat.indices.nbytes + mat.indptr.nbytes
print(matrix_bytes / 1024**3, "GB")

6.828087639063597 GB


In [ ]:
import numpy as np
import anndata as ad

np.random.seed(0)
idx = np.random.choice(mat.shape[1], size=10000, replace=False)

mat_sub = mat[:, idx]
cells_sub = [cells_list[i] for i in idx]

meta_sub = meta.set_index("NAME").loc[cells_sub].copy()

print(mat_sub.shape)
print(meta_sub.shape)

adata = ad.AnnData(
    X=mat_sub.T,   # AnnData 是 cells × genes
    obs=meta_sub,
    var=pd.DataFrame(index=genes_list)
)

print(adata)
print(adata.obs.columns)
print(adata.obs["disease_status"].value_counts())
print(adata.obs["cell_type_anno"].value_counts().head(20))

(33694, 5000)
(5000, 13)
AnnData object with n_obs × n_vars = 5000 × 33694
    obs: 'biosample_id', 'disease', 'disease__ontology_label', 'disease_status', 'donor_id', 'library_preparation_protocol', 'library_preparation_protocol__ontology_label', 'organ', 'organ__ontology_label', 'sex', 'species', 'species__ontology_label', 'cell_type_anno'
Index(['biosample_id', 'disease', 'disease__ontology_label', 'disease_status',
       'donor_id', 'library_preparation_protocol',
       'library_preparation_protocol__ontology_label', 'organ',
       'organ__ontology_label', 'sex', 'species', 'species__ontology_label',
       'cell_type_anno'],
      dtype='object')
disease_status
Active       2682
Remission    1223
Ctrl         1095
Name: count, dtype: int64
cell_type_anno
Suprabasal              1044
Apical cell              852
Basal cell (cycling)     793
CD8+ Teff (KLRC1)        560
Quiescent basal cell     460
Mast cell                372
Treg                     138
Capillary (RGCC)        

In [10]:
adata.write_h5ad("eoe_subset_5k.h5ad")

In [11]:
matrix_bytes = mat.data.nbytes + mat.indices.nbytes + mat.indptr.nbytes
print(matrix_bytes / 1024**3, "GB")

6.82674627751112 GB


In [ ]:
import pandas as pd
from scipy.io import mmread

genes = pd.read_csv("data/EoE_gene_processed.tsv", sep="\t", header=None, encoding='latin-1')
cells = pd.read_csv("data/EoE_cell_processed.tsv", sep="\t", header=None, encoding='latin-1')
meta = pd.read_csv("data/EoE_meta.txt", sep="\t", encoding='latin-1')
meta = meta.iloc[1:].copy()   # 去掉第一行 TYPE/group
meta["NAME"] = meta["NAME"].astype(str)

genes_list = genes[0].astype(str).tolist()
cells_list = cells[0].astype(str).tolist()

print(len(genes_list), len(cells_list), meta.shape)

mat = mmread("data/EoE-processed.mtx").tocsr()
print(mat.shape)


adata_full = ad.AnnData(
    X=mat.T,   # AnnData 是 cells × genes
    obs=meta,
    var=pd.DataFrame(index=genes_list)
)

print(adata_full)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
33694 393763 (393763, 14)


In [9]:
import anndata as ad
print(mat.shape)
adata_full = ad.AnnData(
    X=mat.T,   
    obs=meta,
    var=pd.DataFrame(index=genes_list)
)
print(adata_full)

(33694, 393763)


/opt/miniconda3/envs/mudata/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 393763 × 33694
    obs: 'NAME', 'biosample_id', 'disease', 'disease__ontology_label', 'disease_status', 'donor_id', 'library_preparation_protocol', 'library_preparation_protocol__ontology_label', 'organ', 'organ__ontology_label', 'sex', 'species', 'species__ontology_label', 'cell_type_anno'


In [12]:
adata_full.write_h5ad("eoe.h5ad")

: 

In [3]:
import pandas as pd
from scipy.io import mmread
import anndata as ad

adata = ad.read_h5ad("eoe_subset_5k.h5ad")
adata

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


AnnData object with n_obs × n_vars = 5000 × 33694
    obs: 'biosample_id', 'disease', 'disease__ontology_label', 'disease_status', 'donor_id', 'library_preparation_protocol', 'library_preparation_protocol__ontology_label', 'organ', 'organ__ontology_label', 'sex', 'species', 'species__ontology_label', 'cell_type_anno'

In [4]:
#genes to remove
mt_genes = [g for g in adata.var_names if g.startswith('MT-')]
ribo_genes = [g for g in adata.var_names if g.startswith(('RPL','RPS','MRPS','MRPL'))]
prob_genes = ['MALAT1', 'EEF1A1', 'TPT1']
remove = set(mt_genes + ribo_genes + prob_genes)

features = [g for g in adata.var_names if g not in remove]
print(f"genes before: {adata.n_vars}, after filter: {len(features)}")

genes before: 33694, after filter: 33497


In [6]:
import scanpy as sc
import numpy as np
sc.pp.calculate_qc_metrics(adata, inplace=True)
adata.obs['nFeature_RNA_log2'] = np.log2(adata.obs['n_genes_by_counts'] + 1)

In [7]:
print(adata.obs['disease_status'].value_counts())
print(adata.obs['cell_type_anno'].value_counts())

disease_status
Active       2682
Remission    1223
Ctrl         1095
Name: count, dtype: int64
cell_type_anno
Suprabasal                  1044
Apical cell                  852
Basal cell (cycling)         793
CD8+ Teff (KLRC1)            560
Quiescent basal cell         460
Mast cell                    372
Treg                         138
Capillary (RGCC)             107
Th17                          76
Venous (ACKR1)                70
Fibroblast                    68
Pericyte                      67
Erythroid cell                46
Eosinophil                    33
CD8+ Trm (GZMK)               31
Th2                           28
Plasma cell (IgA+)            23
Arterial (GJA4)               21
NK (XCL1/CD160)               19
CD4+ Trm (CCL5)               18
cDC2B (CD1C)                  16
Mast cell (cycling)           16
Macrophage (SEPP1)            15
CD8+ T cell (MHC II)          14
Macrophage (PLAC8)            13
CD4+ Tcm                      13
Gamma delta T cell            12

In [11]:
import diffxpy.api as de
# ---- 第一步：对每个 cell type 跑 LR DE ----
cell_types = adata.obs['cell_type_anno'].unique()
results = {}

for ct in cell_types:
    # Active vs Ctrl cells of this type
    mask = adata.obs['cell_type_anno'] == ct
    adata_ct = adata[mask & adata.obs['disease_status'].isin(['Active', 'Ctrl']), features]
    
    n_active = (adata_ct.obs['disease_status'] == 'Active').sum()
    n_ctrl   = (adata_ct.obs['disease_status'] == 'Ctrl').sum()
    
    if n_active < 10 or n_ctrl < 10:
        print(f"Skipping {ct}: Active={n_active}, Ctrl={n_ctrl}")
        continue
    
    print(f"Running DE for {ct} (Active={n_active}, Ctrl={n_ctrl})")
    
    # LR test，对应 FindMarkers(test.use='LR', latent.vars='nFeature_RNA_log2')
    test = de.test.lrt(
        adata_ct,
        full_formula_loc  = '~ disease_status + nFeature_RNA_log2',
        reduced_formula_loc = '~ nFeature_RNA_log2'
    )
    
    results[ct] = test.summary()

print("Done:", list(results.keys()))


Running DE for Quiescent basal cell (Active=214, Ctrl=133)


ValueError: type <class 'tuple'> not recognized